# Laya-JA：4コーパス混合の全層LoRA（3シード）と他コーパスでの評価

前回のノートブック（JNLI だけで全層LoRA、公式開発セット 90.6%）の続きです。次の4つを1本で行います。

1. **前回の JNLI 単独モデルを、JSICK・JaNLI・Jamp・JSeM・独自36問で評価する**（比較の基準）
2. **JNLI・JSICK・JaNLI・Jamp の4コーパス混合で学習し直す**（学習率 1e-4、2エポック）
3. **「含意」の取りこぼしを確認する**（ラベル別再現率と、含意の問題がどこへ流れたか）
4. **3シードで回し、調整用データで最良のシードを完成版として保存する**

- 評価用と調整用の問題は `corpus_data.splits` と同じです（JSICK・JaNLI 各512問、Jamp 348問、JSeM 384問、調整用は各192問）。JSeM と独自36問は学習にもモデル選定にも使いません。
- JaNLI は「含意・非含意」の二値データなので、非含意の例は「矛盾または判断できない」の確率の合計で学習・採点します（`corpus_experiment.py` と同じ扱い）。
- 途中でセッションが切れても、終わったシードの結果は保存先に残り、再実行時は読み込むだけで済みます。

**所要時間の目安**：A100 で1シードあたり10分前後、3シードで30〜40分（未実測の概算）。初回は設定セルの `MAX_TRAIN = 256`、`SEEDS = [42]` で一度通すと安全です。

In [ ]:
# 1. GPU の確認
import torch

if not torch.cuda.is_available():
    raise SystemExit("GPU が見つかりません。「ランタイム → ランタイムのタイプを変更」で GPU を選んでください。")
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

In [ ]:
# 2. インストールとリポジトリの取得（前回と同じ。赤い pip の警告は無視してよい）
!pip install -q "transformers==4.57.6"
!pip install -q --no-deps "laya==0.4.31"

import os, sys

REPO_DIR = "/content/laya-ja"
REPO_COMMIT = "3db0cc4007775a5e4aa5445775a1f8a70b198c03"
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/nozomugit/laya-ja.git {REPO_DIR}
!git -C {REPO_DIR} checkout -q {REPO_COMMIT}
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
# 3. 保存先（Google ドライブ）。前回の実行結果（jnli-lora-s42）もここから読む。
SAVE_TO_DRIVE = True

OUT_ROOT = "/content/laya-ja-runs"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_ROOT = "/content/drive/MyDrive/laya-ja-runs"
    except Exception as e:
        print("ドライブを使わずに続行します:", e)
os.makedirs(OUT_ROOT, exist_ok=True)
print("保存先:", OUT_ROOT)

In [ ]:
# 4. 設定
REGIMES = ["mix4"]        # "mix4" = 4コーパス混合。["mix4", "jnli"] にすると、同じ設定での JNLI 単独も学習する（厳密な対照用）
SEEDS = [42, 43, 44]
EPOCHS = 2
ENCODER_LR = 1e-4         # 前回は 2e-4 で1エポック目から過学習したので半分に
HEAD_LR = 1e-4
MICRO_BATCH = 16
GRAD_ACCUM = 2

EXTRA_CAP = 4000          # JSICK・JaNLI・Jamp から学習に使う上限（各コーパス）。None で全件。
                          # JaNLI と Jamp は雛形から作った文が多いので、上限なしだと学習がそちらに偏る
ENTAILMENT_WEIGHT = 1.0   # 「含意」の例の損失に掛ける重み。混合学習でも含意の再現率が低いままなら 1.5〜2.0 を試す
MAX_TRAIN = None          # 動作確認は 256 など（各コーパスの学習例の上限）。None で上の設定どおり

LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
SHUFFLE_OPTIONS = True
PREV_RUN = "jnli-lora-s42"   # 前回の JNLI 単独モデル（保存先の中のフォルダ名）
LOCAL_BASE = None

REGIME_SOURCES = {"mix4": ["jnli", "jsick", "janli", "jamp"], "jnli": ["jnli"]}
assert REGIMES and set(REGIMES) <= set(REGIME_SOURCES) and SEEDS
RUN_CONFIG = {"epochs": EPOCHS, "encoder_lr": ENCODER_LR, "head_lr": HEAD_LR, "micro_batch": MICRO_BATCH,
              "grad_accum": GRAD_ACCUM, "extra_cap": EXTRA_CAP, "entailment_weight": ENTAILMENT_WEIGHT,
              "max_train": MAX_TRAIN, "lora": [LORA_RANK, LORA_ALPHA, LORA_DROPOUT],
              "shuffle_options": SHUFFLE_OPTIONS, "repo_commit": REPO_COMMIT}

In [ ]:
# 5. データ
#    評価用・調整用は corpus_data.splits と同じ。学習用は、評価用・調整用と同じ文の組（前提と仮説を入れ替えたものを含む）を除く。
import json
from collections import Counter
from pathlib import Path

import numpy as np

from corpus_data import SEED as CORPUS_SEED
from corpus_data import load_tsv, normalized, pair_key, shuffled, splits, unique
from corpus_experiment import measures
from experiment import bootstrap_difference, partition
from laya_ja import BASE, REVISION, nli_request

DATA, CORPORA = Path(".cache/data"), Path(".cache/corpora")
TRAIN_FILES = {
    "jsick": "JSICK/jsick/train.tsv",
    "janli": "JaNLI/janli.tsv",
    "jamp": "temporalNLI_dataset/dataset/train_custom-template-hard.tsv",
}

multi = splits(CORPORA, DATA)                      # 3コーパスと JSeM を固定コミットで取得する
d = partition("jnli", DATA, 42, 10**8, 192, 10**8)
jnli_train = d["train"]
used = {r[2] for r in jnli_train[:4096]}
reserved = [r for r in jnli_train[4096:] if r[2] not in used][:256]   # context_experiment.py と同じ取り置き256問
reserved_groups = {r[2] for r in reserved}
probes = json.loads(Path("challenge.json").read_text())

EVALSETS = {
    "jnli": d["eval"], "reserved": reserved,
    "jsick": multi["jsick-eval"], "janli": multi["janli-eval"], "jamp": multi["jamp-eval"],
    "jsem": multi["jsem-eval"],
    "challenge": [(nli_request(p, h), y, cat, str(i)) for i, (cat, p, h, y) in enumerate(probes)],
}
TUNE = {"jnli": d["tune"], **{s: multi[s + "-tune"] for s in TRAIN_FILES}}


def premise(r):
    return normalized(r[0][0].split("\n仮説：")[0])


block = {pair_key(r[0][0]) for rows in [*EVALSETS.values(), *TUNE.values()] for r in rows}
TRAIN = {"jnli": [r for r in jnli_train if r[2] not in reserved_groups and pair_key(r[0][0]) not in block]}
for name, rel in TRAIN_FILES.items():
    rows = unique(shuffled(load_tsv(CORPORA / rel, name, "train"), CORPUS_SEED + 1), block)
    tune_premises = {premise(r) for r in TUNE[name]}
    rows = [r for r in rows if premise(r) not in tune_premises]
    TRAIN[name] = rows[:EXTRA_CAP] if EXTRA_CAP else rows
if MAX_TRAIN:
    TRAIN = {s: rows[:MAX_TRAIN] for s, rows in TRAIN.items()}

NAMES = {0: "含意", 1: "矛盾", 2: "判断できない", 3: "非含意"}
for title, sets in (("学習", TRAIN), ("調整", TUNE), ("評価", EVALSETS)):
    for name, rows in sets.items():
        c = Counter(r[1] for r in rows)
        print(f"{title} {name:10s} {len(rows):6d}件  " + " / ".join(f"{NAMES[k]} {c[k]}" for k in sorted(c)))

In [ ]:
# 6. ベースモデル、入力の作成、採点、LoRA、損失
import hashlib
import math

import laya.train as laya_train
import torch.nn as nn
import torch.nn.functional as F
from laya.common import QTYPES, collate_items, uses_parallel_layout
from laya.train import TrainConfig, encode_item, encode_state, load_checkpoint, make_item, to_internal, train_model
from safetensors import safe_open
from safetensors.torch import load_file, save_file

BASE_SHA256 = "9d628fd971b700382ac6f65920a86f149777b2e748e0c955fb3b19695aa8f204"
if LOCAL_BASE:
    base_dir = LOCAL_BASE
else:
    from huggingface_hub import snapshot_download

    base_dir = snapshot_download(
        BASE, revision=REVISION,
        allow_patterns=["rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*"],
    )
    h = hashlib.sha256()
    with open(os.path.join(base_dir, "model.safetensors"), "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == BASE_SHA256, "ベースモデルのハッシュが laya-ja の固定値と一致しません"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, tok, cfg = load_checkpoint(base_dir)
max_len, head_max_len = cfg.get("max_len", 512), cfg.get("head_max_len", 192)
assert not uses_parallel_layout(cfg)


def load_model(path):
    m, _, _ = load_checkpoint(path)
    return m.to(device).eval()


def make_items(rows, train=False):
    # 目標ベクトルは「正解になりうる選択肢の印 × 重み」。JaNLI の非含意（ラベル3）は矛盾と判断できないの両方に印を付ける。
    items, skipped = [], 0
    for (state, instruction, choices), y, _, _ in rows:
        q = to_internal("decision", {"type": "choice", "instructions": instruction,
                                     "criteria": dict.fromkeys(choices, "")})
        target = [0.0, 1.0, 1.0] if y == 3 else [float(i == y) for i in range(3)]
        if train and y == 0:
            target = [v * ENTAILMENT_WEIGHT for v in target]
        state_ids = encode_state(tok, state, max_len)
        item, reason = make_item(tok, q, target, state_ids, head_max_len, max_len)
        if item is None and not train:
            assert reason == "options_collapsed", reason
            item = {"q": q, "state_ids": state_ids, "target": target, "qtype": QTYPES["choice"], "k": 3}
        if item is None:
            skipped += 1
        else:
            items.append(item)
    if skipped:
        print("学習から除いた例:", skipped)
    return items


def set_likelihood_loss(logits, target, mask):
    # 損失 = 重み × −log(印の付いた選択肢の確率の合計)。印が1つなら通常の交差エントロピーと同じ。
    logp = torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)
    weight = target.max(-1).values
    return -(weight * torch.logsumexp(logp.masked_fill(target <= 0, float("-inf")), -1)).mean()


laya_train.soft_ce_loss = set_likelihood_loss     # 上流の学習ループが呼ぶ損失を差し替える（loss="soft-ce" のとき使われる）


@torch.no_grad()
def score(model, items, batch_size=64):
    was_training = model.training
    model.eval()
    order = sorted(range(len(items)), key=lambda i: len(items[i]["state_ids"]))
    out = [None] * len(items)
    for start in range(0, len(items), batch_size):
        idx = order[start:start + batch_size]
        b = collate_items([[encode_item(tok, items[i], max_len, head_max_len) for i in idx]], tok.pad_token_id)
        logits, _ = model(*(b[k].to(device) for k in
                            ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype")))
        for row, i in zip(logits.float().cpu(), idx):
            out[i] = row[:3]
    model.train(was_training)
    return torch.stack(out)


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.lora_A), self.lora_B) * self.scale

    def merged(self):
        with torch.no_grad():
            delta = (self.lora_B.float() @ self.lora_A.float()) * self.scale
            self.base.weight.add_(delta.to(self.base.weight.dtype))
        return self.base


def add_lora(model, r, alpha, dropout):
    enc = model.encoder
    for p in enc.parameters():
        p.requires_grad_(False)
    for name in [n for n, m in enc.named_modules() if isinstance(m, nn.Linear) and n.startswith("layers.")]:
        parent_name, child = name.rsplit(".", 1)
        parent = enc.get_submodule(parent_name)
        setattr(parent, child, LoRALinear(getattr(parent, child), r, alpha, dropout).to(device))


def merge_lora(model):
    enc = model.encoder
    for name in [n for n, m in enc.named_modules() if isinstance(m, LoRALinear)]:
        parent_name, child = name.rsplit(".", 1)
        parent = enc.get_submodule(parent_name)
        setattr(parent, child, getattr(parent, child).merged())


def apply_adapter(model, path):
    with safe_open(path, framework="pt") as f:
        meta = f.metadata()
    assert meta["base_revision"] == REVISION, "アダプターとベースモデルの版が一致しません"
    add_lora(model, int(meta["rank"]), float(meta["alpha"]), 0.0)
    result = model.load_state_dict(load_file(path), strict=False)
    assert not result.unexpected_keys, result.unexpected_keys
    return model.eval()


EVAL_ITEMS = {name: make_items(rows) for name, rows in EVALSETS.items()}
EVAL_Y = {name: torch.tensor([r[1] for r in rows]) for name, rows in EVALSETS.items()}
TUNE_ITEMS = {name: make_items(rows) for name, rows in TUNE.items()}
TUNE_Y = {name: torch.tensor([r[1] for r in rows]) for name, rows in TUNE.items()}
TRAIN_ITEMS = {name: make_items(rows, train=True) for name, rows in TRAIN.items()}


def measure(name, logits, y=None):
    # JaNLI は含意・非含意の二値で採点する（corpus_experiment.measures と同じ）。戻り値は (指標, 各問の正誤)。
    source = name.split("-")[0]
    return measures(logits, EVAL_Y[name] if y is None else y, source == "janli")


def score_evalsets(model):
    return {name: score(model, items) for name, items in EVAL_ITEMS.items()}


def show(title, scores):
    print(title + "  " + "  ".join(f"{n} {measure(n, s)[0]['accuracy']:.3f}" for n, s in scores.items()))

In [ ]:
# 7. 上流の多言語版（学習前）の成績
model = load_model(base_dir)
SCORES = {"base": score_evalsets(model)}
del model
torch.cuda.empty_cache()
show("上流の多言語版     ", SCORES["base"])

In [ ]:
# 8. 手順1：前回の JNLI 単独モデルを他のコーパスで評価する
PREV_CKPT = os.path.join(OUT_ROOT, PREV_RUN, "checkpoint")
if os.path.exists(os.path.join(PREV_CKPT, "rl_agent_config.json")):
    model = load_model(PREV_CKPT)
    SCORES["prev"] = score_evalsets(model)
    del model
    torch.cuda.empty_cache()
    show("JNLI単独LoRA（前回）", SCORES["prev"])
else:
    print(f"前回のモデルが {PREV_CKPT} に見つからないため、この比較は省きます。")

In [ ]:
# 9. 手順2・4：学習（設定した方式 × シード）
#    各エポックの終わりに、学習に使ったコーパスの調整用データ（各192件）で負の対数尤度を測り、
#    コーパス間で平均した値が最小のエポックを採用する。終わった実行は保存先に残し、再実行時は読み込む。
def run_dir(regime, seed):
    return os.path.join(OUT_ROOT, f"nli-{regime}-lora-s{seed}")


def run(regime, seed):
    out = run_dir(regime, seed)
    done, adapter = os.path.join(out, "run.pt"), os.path.join(out, "adapter.safetensors")
    if os.path.exists(done) and os.path.exists(adapter):
        saved = torch.load(done)
        if saved["config"] == RUN_CONFIG:
            print(f"[{regime} seed {seed}] 保存済みの結果を読み込みました（採用エポック {saved['epoch']}）")
            return saved
    os.makedirs(out, exist_ok=True)
    sources = REGIME_SOURCES[regime]
    items = [it for s in sources for it in TRAIN_ITEMS[s]]
    print(f"[{regime} seed {seed}] 学習例 {len(items)}件", flush=True)
    model = load_model(base_dir)
    add_lora(model, LORA_RANK, LORA_ALPHA, LORA_DROPOUT)
    config = TrainConfig(
        epochs=EPOCHS, micro_batch=MICRO_BATCH, grad_accum=GRAD_ACCUM, encoder_lr=ENCODER_LR, head_lr=HEAD_LR,
        loss="soft-ce", shuffle_options=("choice",) if SHUFFLE_OPTIONS else (),
        gradient_checkpointing=False, seed=seed, log_every=200,
    )
    best = {"nll": float("inf"), "epoch": None, "state": None}
    history = []

    def on_epoch_end(epoch, loss):
        tune = {s: measure(s, score(model, TUNE_ITEMS[s]), TUNE_Y[s])[0] for s in sources}
        nll = sum(m["nll"] for m in tune.values()) / len(tune)
        history.append({"epoch": epoch + 1, "train_loss": loss, "tune_nll": nll,
                        "tune_accuracy": {s: m["accuracy"] for s, m in tune.items()}})
        print(f"  調整用: NLL {nll:.4f} / 正答率 " + ", ".join(f"{s} {m['accuracy']:.3f}" for s, m in tune.items()),
              flush=True)
        if nll < best["nll"]:
            state = {n: p.detach().float().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
            best.update(nll=nll, epoch=epoch + 1, state=state)

    train_model(model, tok, items, config, device, max_len, head_max_len, on_epoch_end=on_epoch_end)
    model.load_state_dict(best["state"], strict=False)
    result = {"config": RUN_CONFIG, "regime": regime, "seed": seed, "epoch": best["epoch"], "tune_nll": best["nll"],
              "history": history, "train_items": len(items), "scores": score_evalsets(model)}
    meta = {"base": BASE, "base_revision": REVISION, "regime": regime, "seed": str(seed),
            "rank": str(LORA_RANK), "alpha": str(LORA_ALPHA), "repo_commit": REPO_COMMIT}
    save_file({k: v.contiguous() for k, v in best["state"].items()}, adapter, metadata=meta)
    torch.save(result, done)
    del model
    torch.cuda.empty_cache()
    show(f"[{regime} seed {seed}] 採用エポック {result['epoch']} ", result["scores"])
    return result


RUNS = {regime: {seed: run(regime, seed) for seed in SEEDS} for regime in REGIMES}

In [ ]:
# 10. まとめ：コーパス別の正答率（シードごと、平均、3シードの採点を平均したアンサンブル）
#     既存の値は TARGET95.md と results/context-nli.json・context-domains.json から。
PUBLISHED = {   # 評価先: (共有の文脈モデル1つ, コーパス別の専用層)
    "jnli": (0.814, 0.848), "reserved": (0.820, 0.824), "jsick": (0.834, 0.840), "janli": (0.893, 0.908),
    "jamp": (0.635, 0.670), "jsem": (0.469, None), "challenge": (0.500, None),
}
for regime in REGIMES:
    SCORES[regime] = {name: torch.stack([RUNS[regime][s]["scores"][name] for s in SEEDS]).mean(0)
                      for name in EVALSETS}

report = {"config": RUN_CONFIG, "seeds": SEEDS, "regimes": REGIMES,
          "counts": {"train": {s: len(v) for s, v in TRAIN_ITEMS.items()},
                     "eval": {s: len(v) for s, v in EVALSETS.items()}},
          "published": PUBLISHED, "models": {}, "runs": {}, "comparisons": {}}
for key, scores in SCORES.items():
    report["models"][key] = {name: measure(name, s)[0] for name, s in scores.items()}
for regime in REGIMES:
    report["runs"][regime] = {
        str(seed): {"epoch": r["epoch"], "tune_nll": r["tune_nll"], "history": r["history"],
                    "train_items": r["train_items"],
                    "evaluation": {name: measure(name, s)[0] for name, s in r["scores"].items()}}
        for seed, r in RUNS[regime].items()
    }


def fmt(v):
    return "  -  " if v is None else f"{100 * v:5.1f}"


main = "mix4" if "mix4" in REGIMES else REGIMES[0]
print(f"正答率（%）。{main} は {len(SEEDS)} シード。JSeM と独自36問（challenge）は学習に使っていない評価先。\n")
header = f"{'評価先':10s} {'件数':>5s} {'上流':>6s} {'既存共有':>7s} {'既存専用':>7s} {'JNLI単独':>8s}"
for regime in REGIMES:
    header += f" | {regime} 平均±SD (最小〜最大) / アンサンブル"
print(header)
for name in EVALSETS:
    acc = lambda key: report["models"][key][name]["accuracy"] if key in report["models"] else None
    line = (f"{name:10s} {len(EVALSETS[name]):5d} {fmt(acc('base')):>6s} {fmt(PUBLISHED[name][0]):>9s} "
            f"{fmt(PUBLISHED[name][1]):>9s} {fmt(acc('prev')):>9s}")
    for regime in REGIMES:
        per_seed = [report["runs"][regime][str(s)]["evaluation"][name]["accuracy"] for s in SEEDS]
        line += (f" | {100 * np.mean(per_seed):5.1f} ±{100 * np.std(per_seed):4.1f} "
                 f"({100 * min(per_seed):.1f}〜{100 * max(per_seed):.1f}) / {fmt(acc(regime))}")
    print(line)

# 同じ問題を対応づけた差の95%区間（グループ単位の再標本化）。アンサンブルを、前回の JNLI 単独・上流と比べる。
print(f"\n{main}（アンサンブル）との差：")
for reference in [k for k in ("prev", "base") if k in SCORES]:
    report["comparisons"][f"{main}_vs_{reference}"] = {}
    for name in EVALSETS:
        diff = measure(name, SCORES[main][name])[1].astype(int) - measure(name, SCORES[reference][name])[1].astype(int)
        lo, hi = bootstrap_difference(diff, np.array([r[2] for r in EVALSETS[name]]))
        report["comparisons"][f"{main}_vs_{reference}"][name] = {
            "difference": float(diff.mean()), "gained": int((diff == 1).sum()), "lost": int((diff == -1).sum()),
            "group_bootstrap_95pct": [lo, hi]}
        print(f"  対 {reference:5s} {name:10s} {100 * diff.mean():+5.1f}pt  (95%区間 {100 * lo:+.1f}〜{100 * hi:+.1f})")

In [ ]:
# 11. 手順3：「含意」の取りこぼしを確認する
#     ラベル別の再現率と、正解が含意の問題をどのラベルに判定したかの内訳（JNLI 公式開発セット）。
LABEL_NAMES = ["含意", "矛盾", "判断できない"]
report["label_recall"] = {}
for name in ("jnli", "jsick", "jamp", "jsem"):
    print(f"\n{name}  ラベル別再現率")
    report["label_recall"][name] = {}
    for key in SCORES:
        recall = measure(name, SCORES[key][name])[0]["recall"]
        report["label_recall"][name][key] = recall
        print(f"  {key:6s} " + "  ".join(f"{LABEL_NAMES[i]} {100 * recall.get(n, float('nan')):5.1f}"
                                         for i, n in enumerate(("entailment", "contradiction", "neutral"))))

print("\nJNLI 公式開発セット：正解が「含意」の問題の判定先")
y = EVAL_Y["jnli"]
report["entailment_predictions"] = {}
for key in SCORES:
    pred = SCORES[key]["jnli"][y == 0].argmax(-1)
    counts = [int((pred == i).sum()) for i in range(3)]
    report["entailment_predictions"][key] = dict(zip(LABEL_NAMES, counts))
    print(f"  {key:6s} " + "  ".join(f"{LABEL_NAMES[i]} {c}" for i, c in enumerate(counts)) + f"  （全{len(pred)}問）")

In [ ]:
# 12. 完成版の保存
#     調整用データの負の対数尤度が最小だったシードを選び（開発セットは選定に使わない）、LoRA を本体に織り込んで保存する。
#     保存後に読み直し、評価が保存前と一致するかを確かめる。
best_seed = min(SEEDS, key=lambda s: RUNS[main][s]["tune_nll"])
print(f"選んだシード: {best_seed}（調整用 NLL {RUNS[main][best_seed]['tune_nll']:.4f}）")
model = apply_adapter(load_model(base_dir), os.path.join(run_dir(main, best_seed), "adapter.safetensors"))
merge_lora(model)

FINAL_DIR = os.path.join(OUT_ROOT, f"nli-{main}-lora-final")
CKPT_DIR = os.path.join(FINAL_DIR, "checkpoint")
os.makedirs(CKPT_DIR, exist_ok=True)
save_file({k: v.detach().half().cpu().contiguous() for k, v in model.state_dict().items()},
          os.path.join(CKPT_DIR, "model.safetensors"))
model.encoder.config.save_pretrained(os.path.join(CKPT_DIR, "encoder"))
tok.save_pretrained(os.path.join(CKPT_DIR, "tokenizer"))
Path(CKPT_DIR, "rl_agent_config.json").write_text(
    json.dumps(dict(cfg, max_len=max_len, head_max_len=head_max_len), indent=2))
del model
torch.cuda.empty_cache()

model = load_model(CKPT_DIR)
reloaded = score_evalsets(model)
del model
torch.cuda.empty_cache()
report["selected_seed"] = best_seed
report["saved_checkpoint_check"] = {}
for name in EVALSETS:
    before = measure(name, RUNS[main][best_seed]["scores"][name])[0]["accuracy"]
    after = measure(name, reloaded[name])[0]["accuracy"]
    report["saved_checkpoint_check"][name] = {"in_memory": before, "reloaded": after}
    flag = "" if abs(before - after) <= 0.005 else "  ← 差が0.5ポイント超。知らせてください"
    print(f"{name:10s} 保存前 {before:.4f} / 読み直し後 {after:.4f}{flag}")
Path(FINAL_DIR, "results.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print("保存先:", FINAL_DIR, sorted(os.listdir(FINAL_DIR)))

In [ ]:
# 13. 保存したモデルを Laya の通常の入口から使ってみる
import laya
from laya_ja import query

agent = laya.load(CKPT_DIR, device=str(device))
examples = [
    ("整理券の所持者は全員入場できる。秋さんは持っていない。", "秋さんは入場できる。", "判断できない"),
    ("月曜に資料を送り、水曜に返事を受け取った。", "資料を送ったのは返事を受け取る前だ。", "含意"),
    ("二人が横断歩道を渡っている。", "人が道を渡っている。", "含意"),
    ("赤い箱と青い箱が一つずつある。両方を開けた。", "青い箱は開けなかった。", "矛盾"),
]
for p, h, gold in examples:
    a = query(agent, *nli_request(p, h))
    print(f"{'○' if a['choice'] == gold else '×'} 正解 {gold} / 判定 {a['choice']} ({a['answer_confidence']:.3f})  {p} → {h}")

## 結果の読み方

- **セル10の表**：列は左から、上流の多言語版、既存の共有文脈モデル（1つのモデルで全コーパス）、既存のコーパス別専用層、前回の JNLI 単独LoRA、今回の混合LoRA です。今回のモデルは評価先を指定せずに1つで全コーパスに答えるので、まず「既存共有」と比べ、そのうえで「既存専用」に届くかを見ます。
- **JSeM と challenge** が、学習していない種類の推論への効き目です。ここが JNLI 単独より上がっていれば、混合学習の狙いどおりです。Jamp（時間関係）と JaNLI（語順・構文）は雛形から作られたデータなので、同じ雛形に慣れただけの伸びが混ざります。
- **シード間のばらつき**：平均±SD と最小〜最大を見てください。方式の差がこの幅より小さい場合、差があるとは言えません。36問の challenge は1問で2.8ポイント動きます。
- **含意の取りこぼし（セル11）**：前回は JNLI の含意の再現率が 64.2% でした。混合後も70%を下回り、流れ先が「判断できない」に偏っているなら、設定セルの `ENTAILMENT_WEIGHT` を 1.5 にして回し直す価値があります（設定を変えると保存済みの結果は使われず、学習し直します）。
- **リポジトリに載せる値**：ここでの評価は GPU 上です。載せる最終値は、保存した `nli-…-final/checkpoint` を CPU で評価し直してください。完成版は調整用データで選んだ1シードのモデルで、アンサンブルの値とは別です。
- **確率の校正はやり直していません**。確信度を閾値に使う場合は、調整用データで温度を合わせ直してください。